## INST326 OOP Project 02

Rename this notebook, replacing "_Assignment" with "_YourName"<br>
Insert Signature Block Here

### Projects
Choose **ONE** of the following projects and write to code solution in the code cell below your choice. Use comments in your code to document your solution. If you need to write comments to the grader, add a markdown cell immediately above your code solution and add your comments there. Be sure to read and follow the Notebook Instructions at the bottom of this notebook. Your grade may depend on it! 

#### General Requirements
>  All of the following options must meet the following requirements.
> 1. All projects must use the "app class template" shown below to organize your code.
> 2. All projects must automatically save content or history generated during use. A simple text file is sufficient, but other formats are OK too. Each program must automatically load the saved content when the app is started.
> 3. Each program must attempt to follow OOP principles, meaning that your attributes and methods are organized into classes. The flow of your program should follow OOP principles of encapsulation and abstraction.

#### Program examples and getting started
>  The AI generated examples included in module 08 are provided as 'prompts' to get you started thinking about your application. None of those examples meet all of the gemeral or specific requirements for the projects listed below. However, they do illustrate one way to solve some programming problems. Study them and then create your own application that meets all the requirements.
>  To get started, you may want to sketch out what your app will look like on paper. Then think about the different parts of your sketch in terms of Tkinter widgets. Which parts are labels? entry boxes? buttons? list boxes? text boxes? check boxes? Think about the layout and the best way to place your widgets in the window. For many apps, using a grid layout is easiest, so think about, and maybe even sketch your app in a grid layout. Also think about whether you need frames to help organize the space in the window. Next, think about functionality. What do your buttons do? What happens when you click a checkbox? How and when are calculations done? What variables or attributes will you need? What methods would best handle your app's behavior? Finally, how might you organize your code into classes according to OOP principles? Notice that a lot of this planning happens with pen and paper, not python code. 

#### 1. GUI Calculator App
>  Create a simple GUI calculator app that meets the following requirements.
> 1. All general requirements listed above.
> 2. Your calculator should include buttons for digits [0-9].
> 3. You should have buttons for the four arithmetic operators [+,-,*,/] and the equal sign [=]
> 4. You should have a button to add a decimal point [.]
> 5. You should have a button to toggle between positive and negative [+/-]
> 6. You program should be able to save and retrieve a history of the last, say, 10 expressions evaluated by the calculator. Recall that an expression is everything you press up to, but not including, the equal sign.
> 7. As discussed in class, the easiest approach to handling this is to use python's eval() function. eval() evaluates the contents of a string *as if* it was a mathematical expression. This makes a lot of the AI generated code in the example unnecessary. (This is not really a requirement. Just a suggestion.)
> 8. Your calculator should have a button to toggle the display of history in your window. (see the lecture images included in module 08). You might think about this in terms of overlapping frames in your window that you can toggle between.
> 9. Create whatever classes you need below the "app class", then call those classes and methods in your app class like we did in example 2 from lecture 6.

#### GUI To-do list app
> Create a simple to-do list app that meets the following requirements.
> 1. All general requirements listed above.
> 2. The abillity to add tasks
> 3. The ability to mark tasks completed.
> 4. The ability to delete tasks.
> 5. While you may follow the format used in the example, the Tkinter checkbox widget would be better.
> 6. The features listed above are all that is requires. You may add additional features if you like. For example, you may want to include due dates.
> 7. Create whatever classes you need below the "app class", then call those classes and methods in your app class like we did in example 2 from lecture 6.

In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
from tkcalendar import Calendar
from datetime import datetime
import json

class Task:
    def __init__(self, description, due_date=None, completed=False):
        self.description = description
        self.due_date = due_date
        self.completed = completed
    
    def to_dict(self):
        return {
            'description': self.description,
            'due_date': self.due_date.strftime('%Y-%m-%d') if self.due_date else None,
            'completed': self.completed
        }
    
    @classmethod
    def from_dict(cls, data):
        due_date = datetime.strptime(data['due_date'], '%Y-%m-%d') if data['due_date'] else None
        return cls(data['description'], due_date, data['completed'])

class TaskManager:
    def __init__(self):
        self.tasks = []
        self.load_tasks()
    
    def add_task(self, description, due_date):
        task = Task(description, due_date)
        self.tasks.append(task)
        self.save_tasks()
    
    def toggle_completion(self, task_index):
        self.tasks[task_index].completed = not self.tasks[task_index].completed
        self.save_tasks()
    
    def delete_task(self, task_index):
        del self.tasks[task_index]
        self.save_tasks()
    
    def save_tasks(self):
        with open('tasks.json', 'w') as f:
            json.dump([task.to_dict() for task in self.tasks], f)
    
    def load_tasks(self):
        try:
            with open('tasks.json', 'r') as f:
                tasks_data = json.load(f)
                self.tasks = [Task.from_dict(task_data) for task_data in tasks_data]
        except FileNotFoundError:
            self.tasks = []

class ToDoWidget(ttk.Frame):
    def __init__(self, parent):
        super().__init__(parent)
        self.task_manager = TaskManager()
        
        # Create the input section
        self.create_input_section()
        
        # Create the task lists sections
        self.create_task_lists()
        
        # Initial refresh
        self.refresh_all()
    
    def create_input_section(self):
        input_frame = ttk.Frame(self)
        input_frame.pack(pady=20)
        
        self.task_entry = ttk.Entry(input_frame, width=40)
        self.task_entry.grid(row=0, column=0, padx=10)
        
        self.calendar = Calendar(input_frame, selectmode='day', date_pattern='yyyy-mm-dd')
        self.calendar.grid(row=0, column=1, padx=10)
        
        add_button = ttk.Button(input_frame, text="Add Task", command=self.add_task)
        add_button.grid(row=0, column=2, padx=10)
    
    def create_task_lists(self):
        # Pending tasks section
        self.pending_frame = ttk.Frame(self)
        self.pending_frame.pack(fill='x', padx=10)
        ttk.Label(self.pending_frame, text="Tasks", font=("Arial", 14)).pack(pady=5)
        
        # Completed tasks section
        self.completed_frame = ttk.Frame(self)
        self.completed_frame.pack(fill='x', padx=10)
        ttk.Label(self.completed_frame, text="Completed Tasks", font=("Arial", 14)).pack(pady=5)
    
    def add_task(self):
        task_desc = self.task_entry.get()
        due_date = self.calendar.get_date()
        
        if task_desc:
            try:
                due_date = datetime.strptime(due_date, '%Y-%m-%d')
                self.task_manager.add_task(task_desc, due_date)
                self.task_entry.delete(0, tk.END)
                self.refresh_all()
            except ValueError:
                messagebox.showerror("Invalid Date", "Please enter a valid date.")
        else:
            messagebox.showerror("Input Error", "Task description cannot be empty.")
    
    def refresh_all(self):
        # Clear existing tasks
        for frame in [self.pending_frame, self.completed_frame]:
            for widget in frame.winfo_children():
                if not isinstance(widget, ttk.Label):  # Preserve the title labels
                    widget.destroy()
        
        # Refresh both sections
        for index, task in enumerate(self.task_manager.tasks):
            target_frame = self.completed_frame if task.completed else self.pending_frame
            
            task_frame = ttk.Frame(target_frame)
            task_frame.pack(fill='x', pady=2)
            
            var = tk.BooleanVar(value=task.completed)
            task_text = f"{task.description} (Due: {task.due_date.strftime('%Y-%m-%d')})"
            
            chk = ttk.Checkbutton(
                task_frame,
                text=task_text,
                variable=var,
                command=lambda idx=index: self.toggle_task(idx))
            chk.pack(side='left')
            
            delete_btn = ttk.Button(
                task_frame,
                text="Delete",
                command=lambda idx=index: self.delete_task(idx))
            delete_btn.pack(side='right')
    
    def toggle_task(self, index):
        self.task_manager.toggle_completion(index)
        self.refresh_all()
    
    def delete_task(self, index):
        self.task_manager.delete_task(index)
        self.refresh_all()

class App(tk.Tk):
    def __init__(self, title, size):
        # main setup
        super().__init__()
        self.title(title)
        self.geometry(f'{size[0]}x{size[1]}')
        self.minsize(size[0], size[1])
        
        # Create and pack the main widget
        self.todo_widget = ToDoWidget(self)
        self.todo_widget.pack(expand=True, fill='both')
        
        # run
        self.mainloop()

if __name__ == "__main__":
    App('To-Do List App', (800, 600))

In [14]:
pip install tkcalendar

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


#### GUI Contact app
> Create a simple contact app that meets the following requirements.
> 1. All general requirements listed above.
> 2. The ability to add contacts.
> 3. The ability to edit contacts.
> 4. The ability to delete contacts.
> 5. The ability to search contacts. The AI example code only does exact match. Your code should be capable of partial matching.
> 6. Minimum fields include name, phone, and email. You may add additional fields if you like.
> 7. You may add additional functionality if you wish, but it is not required. for example, you might include code to format or validate phone numbers or email addresses.
> 8. Create whatever classes you need below the "app class", then call those classes and methods in your app class like we did in example 2 from lecture 6.

#### App Class Example from Lecture 06

In [5]:
# original program source: https://youtu.be/eaxPK9VIkFM?si=84asxl8ZLjPuy9Nv
import tkinter as tk
from tkinter import ttk

class App(tk.Tk):
	def __init__(self, title, size):
		
		# main setup
		super().__init__()
		self.title(title)
		self.geometry(f'{size[0]}x{size[1]}')
		self.minsize(size[0],size[1])

		# widgets
        # this is where you call your other classes
		self.menu = Menu(self) # delete this example in your own code
        self.main = Main(self) # delete this example in your own code

		# run 
		self.mainloop()

# write your own classes below. delete the example classes once you understand how things work.

class Menu(ttk.Frame):
	def __init__(self, parent):
		super().__init__(parent)
		self.place(x = 0, y = 0, relwidth = 0.3, relheight = 1)

		self.create_widgets()

	def create_widgets(self):
		
		# create the widgets 
		menu_button1 = ttk.Button(self, text = 'Button 1')
		menu_button2 = ttk.Button(self, text = 'Button 2')
		menu_button3 = ttk.Button(self, text = 'Button 3')

		menu_slider1 = ttk.Scale(self, orient = 'vertical')
		menu_slider2 = ttk.Scale(self, orient = 'vertical')

		toggle_frame = ttk.Frame(self)
		menu_toggle1 = ttk.Checkbutton(toggle_frame, text = 'check 1')
		menu_toggle2 = ttk.Checkbutton(toggle_frame, text = 'check 2')

		entry = ttk.Entry(self)

		# create the grid
		self.columnconfigure((0,1,2), weight = 1, uniform = 'a')
		self.rowconfigure((0,1,2,3,4), weight = 1, uniform = 'a')

		# place the widgets 
		menu_button1.grid(row = 0, column = 0, sticky = 'nswe', columnspan = 2)
		menu_button2.grid(row = 0, column = 2, sticky = 'nswe')
		menu_button3.grid(row = 1, column = 0, columnspan = 3, sticky = 'nsew')

		menu_slider1.grid(row = 2, column = 0, rowspan = 2, sticky = 'nsew', pady = 20)
		menu_slider2.grid(row = 2, column = 2, rowspan = 2, sticky = 'nsew', pady = 20)

		# toggle layout
		toggle_frame.grid(row = 4, column = 0, columnspan = 3, sticky = 'nsew')
		menu_toggle1.pack(side = 'left', expand = True)
		menu_toggle2.pack(side = 'left', expand = True)

		# entry layout
		entry.place(relx = 0.5, rely = 0.95, relwidth = 0.9, anchor = 'center')		

class Main(ttk.Frame):
	def __init__(self, parent):
		super().__init__(parent)
		self.place(relx = 0.3, y = 0, relwidth = 0.7, relheight = 1)
		Entry(self, 'Entry 1','Button 1','green')
		Entry(self, 'Entry 2','Button 2','blue')
		Entry(self, 'Entry 3','Button 3','green')

class Entry(ttk.Frame):
	def __init__(self, parent, label_text, button_text, label_background):
		super().__init__(parent)

		label = ttk.Label(self, text = label_text, background = label_background)
		button = ttk.Button(self, text = button_text)

		label.pack(expand = True, fill = 'both')
		button.pack(expand = True, fill = 'both', pady = 10)

		self.pack(side = 'left', expand = True, fill = 'both', padx = 20, pady = 20)

# execution code
App('Class based app', (600,600))

TabError: inconsistent use of tabs and spaces in indentation (<string>, line 17)

#### App class template
> Use the template below to write your code. Write you own classes below the comment, then call them in the app class.

In [ ]:
import tkinter as tk
from tkinter import ttk

class App(tk.Tk):
	def __init__(self, title, size):
		
		# main setup
		super().__init__()
		self.title(title)
		self.geometry(f'{size[0]}x{size[1]}')
		self.minsize(size[0],size[1])

		# widgets
        # this is where you call your other classes
		self.menu = Menu(self) # delete this example in your own code


		# run 
		self.mainloop()

# Write your own classes below



# execution code
App('Class based app', (600,600)) #edit to insert your own title and windoe size